# 02 — Company split and EDA

**Purpose.** Create the company-level outer train/test split and produce the descriptive tables and supporting EDA figure.

**Inputs.** `data/processed/model_dataset.csv` from Notebook 01.

**Outputs.** `train.csv`, `test.csv`, split summary, annual failure rates, year distributions, class-feature summaries, and an EDA figure.

**What you will learn.** Why the split is done by company rather than by row, and how the training and final-test samples compare.

**Run first.** Run Notebook 01 first.

## Imports and paths

Notebook 02 loads the prepared modelling table, creates the fixed company-level train/test split, and writes the EDA tables and figure.

In [1]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

def find_project_root(start: Path | None = None) -> Path:
    """Find the project root from the current working directory."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        has_structure = (
            (candidate / "notebooks").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "README.md").is_file()
        )
        if has_structure:
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run the notebook from the repository root or notebooks directory."
    )


PROJECT_ROOT = find_project_root()
project_root = PROJECT_ROOT

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
MODEL_DATASET_PATH = PROCESSED_DATA_DIR / "model_dataset.csv"
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "train.csv"
TEST_DATA_PATH = PROCESSED_DATA_DIR / "test.csv"

for directory in [PROCESSED_DATA_DIR, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


Matplotlib is building the font cache; this may take a moment.


## Split and EDA constants

The split is done by company, and the EDA focuses on the financial variables used in the report figures.

In [2]:
COMPANY_COLUMN = "company_name"
TARGET_COLUMN = "failed"
YEAR_COLUMN = "year"

RANDOM_STATE = 42
OUTER_TEST_SIZE = 0.20

KEY_FEATURES_FOR_EDA = ["X8", "X6", "X11", "X1", "X17", "X15"]

FEATURE_NAME_MAP = {
    "X1": "Current assets",
    "X2": "Cost of goods sold",
    "X3": "Depreciation and amortization",
    "X4": "EBITDA",
    "X5": "Inventory",
    "X6": "Net income",
    "X7": "Total receivables",
    "X8": "Market value",
    "X9": "Net sales",
    "X10": "Total assets",
    "X11": "Total long-term debt",
    "X12": "EBIT",
    "X13": "Gross profit",
    "X14": "Total current liabilities",
    "X15": "Retained earnings",
    "X16": "Total revenue",
    "X17": "Total liabilities",
    "X18": "Total operating expenses",
}


## Split and plotting helpers

These helpers are only the pieces needed for the company split and the EDA figure below.

In [3]:
def create_company_level_split(
    data: pd.DataFrame,
    test_size: float = OUTER_TEST_SIZE,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split rows by company so a firm cannot appear in both samples."""
    missing = {COMPANY_COLUMN, TARGET_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f"Missing required split columns: {sorted(missing)}")

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(
        splitter.split(data, y=data[TARGET_COLUMN], groups=data[COMPANY_COLUMN])
    )
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()


def check_no_company_overlap(left: pd.DataFrame, right: pd.DataFrame) -> bool:
    """Return True when two samples have disjoint company identifiers."""
    left_companies = set(left[COMPANY_COLUMN].unique())
    right_companies = set(right[COMPANY_COLUMN].unique())
    return left_companies.isdisjoint(right_companies)


def create_split_summary(
    full_data: pd.DataFrame,
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
) -> pd.DataFrame:
    """Summarize row counts, company counts, and failure rates for a split."""
    def summarize(name: str, data: pd.DataFrame) -> dict[str, float | int | str]:
        """Return counts and failure rate for one split."""
        return {
            "split": name,
            "n_rows": int(len(data)),
            "n_companies": int(data[COMPANY_COLUMN].nunique()),
            "n_failed_rows": int(data[TARGET_COLUMN].sum()),
            "n_alive_rows": int(len(data) - data[TARGET_COLUMN].sum()),
            "failure_rate": float(data[TARGET_COLUMN].mean()),
        }

    summary = pd.DataFrame(
        [summarize("full", full_data), summarize("train", train_data), summarize("test", test_data)]
    )
    summary["company_share"] = summary["n_companies"] / int(full_data[COMPANY_COLUMN].nunique())
    summary["row_share"] = summary["n_rows"] / int(len(full_data))
    return summary

def apply_project_style() -> None:
    """Apply the Matplotlib style used for project figures."""
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "axes.titlesize": 12,
            "axes.labelsize": 10,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": "#d9d9d9",
            "grid.linewidth": 0.8,
            "lines.linewidth": 2.0,
            "lines.markersize": 5.5,
        }
    )


def style_axis(ax, *, grid_axis: str = "y") -> None:
    """Apply the common grid styling to one axis."""
    ax.grid(axis=grid_axis, linestyle="--", alpha=0.25)


def save_figure(fig, output_path: Path) -> None:
    """Save a figure using the project's export settings."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


## Load prepared modelling data

Read the modelling table created in Notebook 01 and check the rows, companies, and target share before making the outer split.


In [4]:
model_dataset = pd.read_csv(MODEL_DATASET_PATH)
display(model_dataset.head())
display(pd.DataFrame({"rows": [len(model_dataset)], "companies": [model_dataset[COMPANY_COLUMN].nunique()], "failure_rate": [model_dataset[TARGET_COLUMN].mean()]}))

,company_name,year,failed,X1,X2,X3,X4,X5,X6,X7,...,X9,X10,X11,X12,X13,X14,X15,X16,X17,X18
0,C_1,1999,0,511.267,833.107,18.373,89.031,336.018,35.163,128.348,...,1024.333,740.998,180.447,70.658,191.226,163.816,201.026,1024.333,401.483,935.302
1,C_1,2000,0,485.856,713.811,18.577,64.367,320.590,18.531,115.187,...,874.255,701.854,179.987,45.790,160.444,125.392,204.065,874.255,361.642,809.888
2,C_1,2001,0,436.656,526.477,22.496,27.207,286.588,-58.939,77.528,...,638.721,710.199,217.699,4.711,112.244,150.464,139.603,638.721,399.964,611.514
3,C_1,2002,0,396.412,496.747,27.172,30.745,259.954,-12.410,66.322,...,606.337,686.621,164.658,3.573,109.590,203.575,124.106,606.337,391.633,575.592
4,C_1,2003,0,432.204,523.302,26.680,47.491,247.245,3.504,104.661,...,651.958,709.292,248.666,20.811,128.656,131.261,131.884,651.958,407.608,604.467


,rows,companies,failure_rate
0,78682,8971,0.066343


## Create the company-level outer split

The same company must never appear in both the training sample and the final test sample.

In [5]:
train_data, test_data = create_company_level_split(
    model_dataset,
    test_size=OUTER_TEST_SIZE,
    random_state=RANDOM_STATE,
)

assert check_no_company_overlap(train_data, test_data), "Outer split contains company overlap."
assert len(train_data) + len(test_data) == len(model_dataset), "Outer split must preserve all rows."
assert set(train_data[TARGET_COLUMN].unique()) == {0, 1}, "Train split must contain both target classes."
assert set(test_data[TARGET_COLUMN].unique()) == {0, 1}, "Test split must contain both target classes."

split_summary = create_split_summary(model_dataset, train_data, test_data)
assert set(split_summary["split"]) == {"full", "train", "test"}, "Split summary must contain full, train, and test."
assert math.isclose(split_summary["row_share"].sum(), 2.0), "Full, train, and test row shares should sum to 2."

display(split_summary)

,split,n_rows,n_companies,n_failed_rows,n_alive_rows,failure_rate,company_share,row_share
0,full,78682,8971,5220,73462,0.066343,1.000000,1.000000
1,train,62988,7176,4043,58945,0.064187,0.799911,0.800539
2,test,15694,1795,1177,14517,0.074997,0.200089,0.199461


## Save train/test files

The saved split becomes the fixed input for validation and final testing in the rest of the project.


In [6]:
train_data.to_csv(TRAIN_DATA_PATH, index=False)
test_data.to_csv(TEST_DATA_PATH, index=False)
split_summary.to_csv(TABLES_DIR / "split_summary.csv", index=False)

saved_split_files = pd.DataFrame(
    {
        "file": ["data/processed/train.csv", "data/processed/test.csv", "outputs/tables/split_summary.csv"],
        "exists": [TRAIN_DATA_PATH.exists(), TEST_DATA_PATH.exists(), (TABLES_DIR / "split_summary.csv").exists()],
    }
)
display(saved_split_files)

,file,exists
0,data/processed/train.csv,True
1,data/processed/test.csv,True
2,outputs/tables/split_summary.csv,True


## Annual failure rate

This section creates the annual-failure-rate table. Notebook 06 later converts that table into a paper-ready figure.


In [7]:
annual_failure_rate = model_dataset.groupby(YEAR_COLUMN)[TARGET_COLUMN].agg(
    n_observations="count",
    n_failed="sum",
).reset_index()
annual_failure_rate["n_alive"] = annual_failure_rate["n_observations"] - annual_failure_rate["n_failed"]
annual_failure_rate["failure_rate"] = annual_failure_rate["n_failed"] / annual_failure_rate["n_observations"]
annual_failure_rate.to_csv(TABLES_DIR / "annual_failure_rate.csv", index=False)
display(annual_failure_rate)

,year,n_observations,n_failed,n_alive,failure_rate
0,1999,5308,380,4928,0.071590
1,2000,5226,404,4822,0.077306
2,2001,4897,414,4483,0.084542
3,2002,4651,414,4237,0.089013
4,2003,4417,415,4002,0.093955
5,2004,4348,404,3944,0.092916
6,2005,4205,379,3826,0.090131
7,2006,4128,366,3762,0.088663
8,2007,4009,336,3673,0.083811
9,2008,3857,284,3573,0.073632


## Train/test year distribution

This table checks that the company-level split still covers the sample period in both the training and test files.


In [8]:
train_years = train_data.groupby(YEAR_COLUMN).size().reset_index(name="n_observations").assign(split="train")
test_years = test_data.groupby(YEAR_COLUMN).size().reset_index(name="n_observations").assign(split="test")
year_distribution = pd.concat([train_years, test_years], ignore_index=True)
year_distribution["share_within_split"] = year_distribution["n_observations"] / year_distribution.groupby("split")["n_observations"].transform("sum")
year_distribution = year_distribution[["split", YEAR_COLUMN, "n_observations", "share_within_split"]]
year_distribution.to_csv(TABLES_DIR / "train_test_year_distribution.csv", index=False)
display(year_distribution.head(10))
display(year_distribution.tail(10))

,split,year,n_observations,share_within_split
0,train,1999,4231,0.067172
1,train,2000,4145,0.065806
2,train,2001,3899,0.061901
3,train,2002,3702,0.058773
4,train,2003,3515,0.055804
5,train,2004,3469,0.055074
6,train,2005,3364,0.053407
7,train,2006,3303,0.052439
8,train,2007,3219,0.051105
9,train,2008,3092,0.049089


,split,year,n_observations,share_within_split
30,test,2009,744,0.047407
31,test,2010,719,0.045814
32,test,2011,686,0.043711
33,test,2012,684,0.043584
34,test,2013,692,0.044093
35,test,2014,684,0.043584
36,test,2015,644,0.041035
37,test,2016,627,0.039952
38,test,2017,589,0.037530
39,test,2018,518,0.033006


## Key-feature summary by status

These median comparisons show how selected financial ratios differ between alive and failed observations.


In [9]:
rows = []
for feature in KEY_FEATURES_FOR_EDA:
    for status_value, status_name in [(0, "alive"), (1, "failed")]:
        subset = model_dataset.loc[model_dataset[TARGET_COLUMN] == status_value, feature]
        rows.append(
            {
                "feature": feature,
                "readable_name": FEATURE_NAME_MAP.get(feature, feature),
                "status": status_name,
                "mean": subset.mean(),
                "median": subset.median(),
                "std": subset.std(),
                "n_observations": int(subset.shape[0]),
            }
        )
class_feature_summary = pd.DataFrame(rows)
class_feature_summary.to_csv(TABLES_DIR / "class_feature_summary.csv", index=False)
display(class_feature_summary)

,feature,readable_name,status,mean,median,std,n_observations
0,X8,Market value,alive,3596.015157,240.89525,19022.527429,73462
1,X8,Market value,failed,857.813011,117.79940,3396.453790,5220
2,X6,Net income,alive,141.994726,2.07150,1299.259556,73462
3,X6,Net income,failed,-48.112333,-3.32700,592.050918,5220
4,X11,Total long-term debt,alive,730.516984,7.09250,3309.223955,73462
5,X11,Total long-term debt,failed,609.430007,18.43850,2077.661309,5220
6,X1,Current assets,alive,914.542615,102.91750,4052.047889,73462
7,X1,Current assets,failed,399.339353,75.87150,1147.837282,5220
8,X17,Total liabilities,alive,1809.571974,80.74050,8257.442726,73462
9,X17,Total liabilities,failed,1266.816748,97.03500,4221.179588,5220


## Relative median-gap figure

The figure turns the median gaps into a quick visual check of which variables separate the two classes most clearly.


In [10]:
medians = class_feature_summary.pivot(index=["feature", "readable_name"], columns="status", values="median")
plot_data = medians.copy()
relative_scale = (plot_data["failed"].abs() + plot_data["alive"].abs()) / 2
plot_data["relative_median_difference"] = (plot_data["failed"] - plot_data["alive"]) / relative_scale.replace(0, pd.NA)
plot_data = plot_data.reset_index()
plot_data["readable_name"] = pd.Categorical(
    plot_data["readable_name"],
    categories=[FEATURE_NAME_MAP.get(feature, feature) for feature in KEY_FEATURES_FOR_EDA],
    ordered=True,
)
plot_data = plot_data.sort_values("readable_name")

apply_project_style()
fig, ax = plt.subplots(figsize=(8.2, 4.8))
colors = ["#c44e52" if value > 0 else "#4c78a8" for value in plot_data["relative_median_difference"]]
bars = ax.barh(plot_data["readable_name"], plot_data["relative_median_difference"], color=colors)
max_abs_difference = plot_data["relative_median_difference"].abs().max()
label_offset = max_abs_difference * 0.04

ax.set_title("Relative Median Gap by Bankruptcy Status", pad=22)
ax.text(0.5, 1.01, "Descriptive comparison only; values do not imply causality.", transform=ax.transAxes, ha="center", va="bottom", fontsize=8.5, color="#4d4d4d")
ax.set_xlabel("Difference in group medians divided by average absolute magnitude")
ax.set_ylabel("Financial variable")
ax.axvline(0, linewidth=1, color="black")
ax.set_xlim(-max_abs_difference * 1.25, max_abs_difference * 1.25)
style_axis(ax, grid_axis="x")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
for bar, value in zip(bars, plot_data["relative_median_difference"], strict=False):
    ax.text(value + (label_offset if value >= 0 else -label_offset), bar.get_y() + bar.get_height() / 2, f"{value:.0%}", va="center", ha="left" if value >= 0 else "right", fontsize=8)

save_figure(fig, FIGURES_DIR / "key_feature_median_by_status.png")
assert (FIGURES_DIR / "key_feature_median_by_status.png").exists(), "EDA figure was not saved."
display(plot_data[["feature", "readable_name", "relative_median_difference"]])

status,feature,readable_name,relative_median_difference
5,X8,Market value,-0.686355
4,X6,Net income,-2.000000
1,X11,Total long-term debt,0.888802
0,X1,Current assets,-0.302547
3,X17,Total liabilities,0.183315
2,X15,Retained earnings,-1.973129


## Summary

- Created a company-level outer split, so no firm appears in both the training and final test samples.
- Saved `data/processed/train.csv`, `data/processed/test.csv`, and the split summary table.
- Produced the main EDA outputs: annual failure rate, train/test year distribution, class feature summaries, and the key-feature median figure.
- Notebook 03 uses only the training file from this split for model selection.
